In [1]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

# 1. Configuración de rutas
dir_datos = "../../Datos/Datasets Finales"

print("=== ESTUDIO DE ABLACIÓN: CORRECCIÓN DE CIRCULARIDAD DEFINICIONAL ===")

# Variables a EXCLUIR para evitar la trampa del GRD (las solicitadas por el profe)
vars_circulares = ['NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS', 'DIAS_ESTADIA']
# Nota: COMORBILIDAD_PRINCIPAL ya está en formato One-Hot, así que buscaremos todas sus columnas.

# 2. Cargar datos oncológicos de Entrenamiento y Prueba
print("Cargando datasets oncológicos...")
df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

# Identificar todas las columnas OHE de COMORBILIDAD_PRINCIPAL
cols_comorbilidad = [col for col in df_onco_train.columns if col.startswith('COMORBILIDAD_PRINCIPAL_')]
todas_vars_prohibidas = vars_circulares + cols_comorbilidad

print(f"Se eliminarán {len(todas_vars_prohibidas)} variables asociadas al cálculo interno del GRD.")

def entrenar_evaluar_estricto(target_name, df_train, df_test, prohibidas):
    print(f"\n--- Entrenando Modelo Clínico Estricto: {target_name} ---")
    
    # Preparar X e y (Eliminando targets y variables prohibidas)
    cols_drop = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CIP_ENCRIPTADO', 'CATEGORIA_CANCER'] + prohibidas
    
    X_train = df_train.drop(columns=cols_drop, errors='ignore')
    y_train = df_train[target_name]
    
    X_test = df_test.drop(columns=cols_drop, errors='ignore')
    # Alinear columnas por si acaso
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_test = df_test[target_name]
    
    # Entrenar XGBoost con los hiperparámetros óptimos que ya tenías
    modelo = xgb.XGBClassifier(
        learning_rate=0.3, 
        max_depth=10, 
        tree_method='hist', 
        n_jobs=-1, 
        random_state=42
    )
    modelo.fit(X_train, y_train)
    
    # Evaluar
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)
    clases = np.unique(y_test)
    y_test_bin = label_binarize(y_test, classes=clases)
    
    f1_macro = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    auprc = average_precision_score(y_test_bin, y_prob, average='weighted')
    
    print(f"F1-Macro Estricto: {f1_macro:.4f}")
    print(f"AUC-ROC Estricto:  {auc:.4f}")
    print(f"AUPRC Estricto:    {auprc:.4f}")
    
    return f1_macro, auc, auprc

# 3. Ejecutar para Severidad
f1_sev, auc_sev, auprc_sev = entrenar_evaluar_estricto('SEVERIDAD', df_onco_train, df_onco_test, todas_vars_prohibidas)

# 4. Ejecutar para Consumo de Recursos
f1_cons, auc_cons, auprc_cons = entrenar_evaluar_estricto('CONSUMO_RECURSOS', df_onco_train, df_onco_test, todas_vars_prohibidas)

=== ESTUDIO DE ABLACIÓN: CORRECCIÓN DE CIRCULARIDAD DEFINICIONAL ===
Cargando datasets oncológicos...
Se eliminarán 20 variables asociadas al cálculo interno del GRD.

--- Entrenando Modelo Clínico Estricto: SEVERIDAD ---
F1-Macro Estricto: 0.6716
AUC-ROC Estricto:  0.8538
AUPRC Estricto:    0.7161

--- Entrenando Modelo Clínico Estricto: CONSUMO_RECURSOS ---
F1-Macro Estricto: 0.6799
AUC-ROC Estricto:  0.8515
AUPRC Estricto:    0.8158


In [3]:
import joblib
# 1. Configuración de rutas
dir_datos = "../../Datos/Datasets Finales"
dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
os.makedirs(dir_modelos, exist_ok=True)

print("=== ESTUDIO DE ABLACIÓN: ENTRENANDO Y GUARDANDO MODELOS ESTRICTOS ===")

# Variables a EXCLUIR
vars_circulares = ['NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS', 'DIAS_ESTADIA']

# 2. Cargar datos oncológicos
print("Cargando datasets oncológicos...")
df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

cols_comorbilidad = [col for col in df_onco_train.columns if col.startswith('COMORBILIDAD_PRINCIPAL_')]
todas_vars_prohibidas = vars_circulares + cols_comorbilidad

def entrenar_guardar_estricto(target_name, df_train, df_test, prohibidas):
    print(f"\n--- Entrenando Modelo Clínico Estricto: {target_name} ---")
    
    cols_drop = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CIP_ENCRIPTADO', 'CATEGORIA_CANCER'] + prohibidas
    
    X_train = df_train.drop(columns=cols_drop, errors='ignore')
    y_train = df_train[target_name]
    
    X_test = df_test.drop(columns=cols_drop, errors='ignore')
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_test = df_test[target_name]
    
    # Entrenar
    modelo = xgb.XGBClassifier(
        learning_rate=0.3, 
        max_depth=10, 
        tree_method='hist', 
        n_jobs=-1, 
        random_state=42
    )
    modelo.fit(X_train, y_train)
    
    # GUARDAR EL MODELO ESTRICTO
    ruta_modelo = os.path.join(dir_modelos, f"Modelo_Optimo_XGBoost_{target_name}_ESTRICTO.pkl")
    joblib.dump(modelo, ruta_modelo)
    print(f"-> Modelo guardado exitosamente en: {ruta_modelo}")
    
    # Evaluar rápido para confirmar
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)
    clases = np.unique(y_test)
    y_test_bin = label_binarize(y_test, classes=clases)
    
    f1_macro = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    
    print(f"F1-Macro Estricto: {f1_macro:.4f} | AUC-ROC: {auc:.4f}")

# 3. Ejecutar y guardar
entrenar_guardar_estricto('SEVERIDAD', df_onco_train, df_onco_test, todas_vars_prohibidas)
entrenar_guardar_estricto('CONSUMO_RECURSOS', df_onco_train, df_onco_test, todas_vars_prohibidas)

print("\n¡Modelos estrictos generados y guardados!")

=== ESTUDIO DE ABLACIÓN: ENTRENANDO Y GUARDANDO MODELOS ESTRICTOS ===
Cargando datasets oncológicos...

--- Entrenando Modelo Clínico Estricto: SEVERIDAD ---
-> Modelo guardado exitosamente en: ../../Resultados/Resultados (etapa 3 y 4)/XGBoost/Modelo_Optimo_XGBoost_SEVERIDAD_ESTRICTO.pkl
F1-Macro Estricto: 0.6716 | AUC-ROC: 0.8538

--- Entrenando Modelo Clínico Estricto: CONSUMO_RECURSOS ---
-> Modelo guardado exitosamente en: ../../Resultados/Resultados (etapa 3 y 4)/XGBoost/Modelo_Optimo_XGBoost_CONSUMO_RECURSOS_ESTRICTO.pkl
F1-Macro Estricto: 0.6799 | AUC-ROC: 0.8515

¡Modelos estrictos generados y guardados!
